# Satellite Mask R-CNN — iSAID on Google Colab

End-to-end on a Colab GPU (free T4 = 16 GB, ~4–8× faster than a 4 GB laptop GPU): **tile → train → evaluate → predict**. Tiling and training run on Colab's fast local disk; **checkpoints, logs, and predictions are saved to Google Drive**, so if Colab disconnects you just re-run the cells and training resumes from the last epoch.

## One-time setup before running
1. Enable a GPU: **Runtime -> Change runtime type -> Hardware accelerator: GPU**.
2. Upload your project folder to Google Drive as `MyDrive/satellite_maskrcnn/` containing:
   - `src/`  and  `configs/config.yaml`  (the repo code)
   - `data/annotations/iSAID_train.json`
   - the **raw training images**, either already extracted in `data/images/` (P0000.png ...) **or** as `part1.zip` / `part2.zip` / `part3.zip` placed in `data/raw_zips/`.

Then run the cells top to bottom: cells 1–6 set up data, **cell 7 trains**, **cell 8 evaluates** the best checkpoint on the full val set, and **cells 9–10 run prediction** and show the result. After a disconnect, just run them all again — tiling is cached and training resumes from the last epoch.

In [ ]:
# 1) GPU check + mount Google Drive
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('NO GPU -> Runtime > Change runtime type > GPU')

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Paths. EDIT DRIVE_PROJECT only if you uploaded the folder somewhere else.
import os, shutil

DRIVE_PROJECT = '/content/drive/MyDrive/satellite_maskrcnn'   # <-- your uploaded folder on Drive
WORK          = '/content/work'                               # fast local working copy
OUT_DIR       = os.path.join(DRIVE_PROJECT, 'outputs')        # checkpoints/logs persist here on Drive

assert os.path.isdir(DRIVE_PROJECT), f'Not found on Drive: {DRIVE_PROJECT}'
os.makedirs(WORK, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# Copy code to local disk (fast); keep big data referenced from Drive as needed.
for sub in ['src', 'configs']:
    src = os.path.join(DRIVE_PROJECT, sub)
    dst = os.path.join(WORK, sub)
    assert os.path.isdir(src), f'Missing {src} on Drive'
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(src, dst)
os.makedirs(os.path.join(WORK, 'data', 'annotations'), exist_ok=True)
print('Code copied to', WORK)

In [ ]:
# 3) Install the few deps Colab doesn't ship (torch/torchvision/opencv/tqdm are preinstalled)
!pip -q install pycocotools albumentations pyyaml

In [ ]:
# 4) Patch the config for Colab: bigger batch (16 GB GPU), local patches, Drive-backed outputs.
import yaml
cfg_path = os.path.join(WORK, 'configs', 'config.yaml')
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['raw_images_dir'] = os.path.join(WORK, 'data', 'images')
cfg['dataset']['raw_ann_file']   = os.path.join(WORK, 'data', 'annotations', 'iSAID_train.json')
cfg['dataset']['images_dir']     = os.path.join(WORK, 'data', 'patches', 'images')
cfg['dataset']['ann_file']       = os.path.join(WORK, 'data', 'patches', 'annotations.json')
cfg['dataset']['split_indices_path'] = os.path.join(OUT_DIR, 'splits', 'train_val_split.json')

# 16 GB GPU: real batch 4 x accum 2 = effective batch 8. No accumulation tricks needed for memory.
cfg['training']['batch_size']         = 4
cfg['training']['accumulation_steps'] = 2
cfg['training']['num_workers']        = 2
cfg['training']['checkpoint_dir']     = os.path.join(OUT_DIR, 'checkpoints')

# Validation knobs (these defend a 4 GB laptop; on a 16 GB T4 they're optional).
# - val_fraction: validate on 25% of the val set each epoch for speed. The T4 has
#   the memory to raise this to 1.0 for full-set monitoring if you prefer.
# - val_eval_fp16: MUST stay False -- fp16 breaks torchvision's mask post-processing
#   and collapses mAP to 0. The final number comes from evaluate.py (full fp32).
cfg['training']['val_fraction']    = 0.25
cfg.setdefault('inference', {})['val_eval_fp16'] = False

cfg['logging']['log_file']        = os.path.join(OUT_DIR, 'training_log.csv')
cfg['logging']['plot_file']       = os.path.join(OUT_DIR, 'training_plot.png')
cfg['logging']['val_preview_dir'] = os.path.join(OUT_DIR, 'validation_previews')

with open(cfg_path, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print('Config patched. batch', cfg['training']['batch_size'],
      'x accum', cfg['training']['accumulation_steps'],
      '| val_fraction', cfg['training']['val_fraction'],
      '| checkpoints ->', cfg['training']['checkpoint_dir'])

In [ ]:
# 5) Get the raw images onto local disk (extract zips if needed) and copy annotations.
import glob, zipfile

img_dir = os.path.join(WORK, 'data', 'images')
os.makedirs(img_dir, exist_ok=True)

# annotations
src_ann = os.path.join(DRIVE_PROJECT, 'data', 'annotations', 'iSAID_train.json')
assert os.path.isfile(src_ann), f'Missing {src_ann}'
shutil.copy(src_ann, os.path.join(WORK, 'data', 'annotations', 'iSAID_train.json'))

# images: prefer an already-extracted Drive folder, else extract part*.zip
drive_imgs = os.path.join(DRIVE_PROJECT, 'data', 'images')
zips = sorted(glob.glob(os.path.join(DRIVE_PROJECT, 'data', 'raw_zips', '*.zip')))
have_local = len(glob.glob(os.path.join(img_dir, 'P*.png')))
if have_local >= 1400:
    print('Images already present locally:', have_local)
elif zips:
    for z in zips:
        print('Extracting', os.path.basename(z), '...')
        with zipfile.ZipFile(z) as zf:
            for m in zf.namelist():
                if m.lower().endswith('.png'):
                    m_dst = os.path.join(img_dir, os.path.basename(m))
                    with zf.open(m) as s, open(m_dst, 'wb') as d:
                        shutil.copyfileobj(s, d)
elif os.path.isdir(drive_imgs):
    print('Copying extracted images from Drive (may be slow)...')
    for p in glob.glob(os.path.join(drive_imgs, 'P*.png')):
        shutil.copy(p, img_dir)
else:
    raise FileNotFoundError('No images found: put part*.zip in data/raw_zips/ or P*.png in data/images/ on Drive')
print('Local raw images:', len(glob.glob(os.path.join(img_dir, 'P*.png'))))

In [ ]:
# 6) Tile the dataset. Cached to Drive (patches.zip) so reconnects don't re-tile.
%cd $WORK
patches_ann = os.path.join(WORK, 'data', 'patches', 'annotations.json')
drive_patches_zip = os.path.join(DRIVE_PROJECT, 'data', 'patches.zip')

if os.path.isfile(patches_ann):
    print('Patches already present locally.')
elif os.path.isfile(drive_patches_zip):
    print('Restoring tiles from Drive cache...')
    shutil.unpack_archive(drive_patches_zip, os.path.join(WORK, 'data', 'patches'))
else:
    !python src/prepare_isaid.py
    print('Zipping tiles to Drive cache (one-time, a few minutes)...')
    shutil.make_archive(os.path.join(DRIVE_PROJECT, 'data', 'patches'), 'zip',
                        os.path.join(WORK, 'data', 'patches'))
print('Tiles ready:', os.path.isfile(patches_ann))

In [ ]:
# 7) Train. Resumes automatically from outputs/checkpoints/last.pth on Drive after any disconnect.
%cd $WORK
!python src/train.py

In [ ]:
# 8) (Optional) Evaluate the best checkpoint
%cd $WORK
!python src/evaluate.py "$OUT_DIR/checkpoints/model_best.pth"

## Predict with the trained model
Run instance segmentation with `model_best.pth`. `predict.py` automatically tiles a large satellite scene (800 px tiles, 200 px overlap) and stitches the predictions back together, so you can pass a full-size image or a folder of images.

- **Overlay** (`*_overlay.png`) and **JSON** (`*_predictions.json`) are written to `outputs/results/` on Drive.
- Set `PRED_INPUT` in the next cell to your own image; leave it blank to auto-pick a sample tile as a sanity check.
- Want pixel masks too? Add `--save-instance-mask` and/or `--save-class-mask` to the command.

In [ ]:
# 9) Run prediction with the trained model. Saves an overlay + JSON to Drive.
#    predict.py tiles a large scene (800 px tiles, 200 px overlap) and stitches it,
#    so PRED_INPUT can be a full-size satellite image OR a folder of images.
%cd $WORK
import glob

BEST        = os.path.join(OUT_DIR, 'checkpoints', 'model_best.pth')
PRED_INPUT  = ''                                # <-- set to your image/folder, e.g. a scene on Drive
PRED_OUTPUT = os.path.join(OUT_DIR, 'results')  # predictions persist to Drive

assert os.path.isfile(BEST), f'No trained checkpoint at {BEST} -- run training (cell 7) first.'
if not PRED_INPUT:
    cand = sorted(glob.glob(os.path.join(WORK, 'data', 'patches', 'images', '*.png')))
    assert cand, 'No sample tiles found; set PRED_INPUT to an image path.'
    PRED_INPUT = cand[0]
    print('No PRED_INPUT set; using sample tile:', PRED_INPUT)

!python src/predict.py --checkpoint "$BEST" --input "$PRED_INPUT" --output "$PRED_OUTPUT" --save-overlay --save-json

In [ ]:
# 10) Show the most recent prediction overlay inline.
import glob
import matplotlib.pyplot as plt
from PIL import Image

overlays = sorted(glob.glob(os.path.join(PRED_OUTPUT, '*_overlay.png')), key=os.path.getmtime)
assert overlays, f'No *_overlay.png in {PRED_OUTPUT} -- run the predict cell first.'
latest = overlays[-1]
print('Showing:', latest)
plt.figure(figsize=(12, 12))
plt.imshow(Image.open(latest))
plt.axis('off')
plt.title(os.path.basename(latest))
plt.show()

## If Colab disconnects
Free Colab drops idle sessions (~90 min) and caps runtime (~12 h). That's fine here:
- `outputs/checkpoints/last.pth` lives on **Drive**, so progress is never lost.
- Just reopen the notebook and **run all cells again**. Cell 6 restores tiles from the Drive cache (no re-tiling), and cell 7's `train.py` resumes from the last completed epoch.
- Keep the browser tab active to reduce idle disconnects. For uninterrupted multi-hour runs consider **Colab Pro** (longer sessions + background execution).

Monitor `outputs/training_log.csv` (Drive) for mAP@50 / mask IoU per epoch; `model_best.pth` is the best model so far and is usable any time.